# Notebook 2: Data Preprocessing & Feature Engineering
## AI-Powered Smart University Assistant

This notebook covers:
- Handling missing values
- Encoding categorical variables
- VLE and Assessment feature engineering
- Merging all OULAD files
- Class imbalance analysis
- Saving processed dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
sys.path.append('..')
from src.preprocessing import load_data, preprocess_and_engineer_features, save_processed_data, generate_early_warning_data
warnings = __import__('warnings'); warnings.filterwarnings('ignore')
print('Libraries loaded.')

## 1. Load Raw Data

In [ ]:
info, assessment, assessments_df, vle, registration = load_data('../data/raw')
print('Files loaded:', [x.shape for x in [info, assessment, assessments_df, vle, registration]])

## 2. Missing Value Analysis & Handling

In [ ]:
print('Missing values in studentInfo:')
print(info.isnull().sum()[info.isnull().sum() > 0])
print('\nStrategy: Fill imd_band NaN with "Missing" category')
info['imd_band'].fillna('Missing', inplace=True)
print('After filling:', info['imd_band'].isnull().sum(), 'remaining nulls')

## 3. Feature Engineering

We engineer the following features:
1. **total_vle_clicks** — Sum of all VLE interactions
2. **active_learning_days** — Number of unique days with activity
3. **avg_clicks_per_active_day** — Engagement intensity
4. **assessments_attempted** — Number of assessments submitted
5. **average_assessment_score** — Mean score across assessments
6. **avg_submission_delay** — Average days early/late
7. **avg_registration_delay** — Registration timing

In [ ]:
# VLE Feature Engineering
vle_agg = vle.groupby('id_student').agg(
    total_vle_clicks=('sum_click', 'sum'),
    active_learning_days=('date', 'nunique')
).reset_index()
vle_agg['avg_clicks_per_active_day'] = vle_agg['total_vle_clicks'] / vle_agg['active_learning_days']
vle_agg['avg_clicks_per_active_day'].fillna(0, inplace=True)
print('VLE features engineered:')
print(vle_agg.describe())

In [ ]:
# Assessment Feature Engineering
assess_merged = pd.merge(assessment, assessments_df, on='id_assessment', how='left')
assess_merged['score'].fillna(0, inplace=True)
assess_merged['submission_delay'] = assess_merged['date_submitted'] - assess_merged['date']
assess_agg = assess_merged.groupby('id_student').agg(
    assessments_attempted=('id_assessment', 'count'),
    average_assessment_score=('score', 'mean'),
    avg_submission_delay=('submission_delay', 'mean')
).reset_index()
print('Assessment features engineered:')
print(assess_agg.describe())

In [ ]:
# Run full preprocessing
processed_df = preprocess_and_engineer_features(info, assessment, assessments_df, vle, registration)
print('\nFinal dataset shape:', processed_df.shape)
print('\nFeature list:')
print(processed_df.columns.tolist())

## 4. Class Imbalance Analysis

In [ ]:
at_risk_counts = processed_df['At_Risk'].value_counts()
print('Class Distribution:')
print(at_risk_counts)
print(f'\nAt-Risk Rate: {at_risk_counts[1] / len(processed_df) * 100:.1f}%')
print(f'Not At-Risk Rate: {at_risk_counts[0] / len(processed_df) * 100:.1f}%')
print('\nNote: Class imbalance is moderate. Strategy: class_weight="balanced" in models.')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Not At Risk', 'At Risk'], at_risk_counts.values, color=['#10b981', '#ef4444'])
ax.set_title('Class Distribution (At_Risk Target Variable)')
ax.set_ylabel('Number of Students')
for i, v in enumerate(at_risk_counts.values):
    ax.text(i, v + 100, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../screenshots/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Generate Early Warning Dataset

In [ ]:
early_df = generate_early_warning_data(info, assessment, assessments_df, vle, cutoff_days=30)
# Add target
early_df['At_Risk'] = early_df['final_result'].map({'Fail': 1, 'Withdrawn': 1, 'Pass': 0, 'Distinction': 0})
print('Early warning dataset shape:', early_df.shape)
save_processed_data(early_df, '../data/processed/early_warning_features.csv')
save_processed_data(processed_df, '../data/processed/student_features.csv')
print('Datasets saved!')

## Summary
- 7 engineered features created from 5 OULAD files
- Missing values in `imd_band` filled with 'Missing' category
- Missing scores treated as 0
- Class imbalance: 52.8% at-risk vs 47.2% not at-risk — addressed with `class_weight='balanced'`
- Both full-semester and 30-day early warning datasets saved